# Mangrove Pulse: Khor Kalba inspection handoff

Prepared 7 October 2026. Team: Taimaa Ghazal (Team Leader & Concept Development Lead), Salim Ammar (Technical Lead & Presenter). Theme: Ecosystem Health, Biodiversity & Blue Carbon.

This notebook turns existing satellite evidence into proposed observation requests. It exports a handoff rather than running a drone, acquiring imagery, recomputing geospatial results or supplying field truth. Read the [executed end-to-end notebook](../Mangrove_Pulse_End_to_End_Executed.ipynb) for the main computational demonstration; the original source is `Mangrove_Pulse.ipynb`.

Run all cells to validate the embedded handoff and download its ZIP. Export checks establish format consistency, not scientific accuracy or engineering feasibility.

## What the drone would investigate

| Case | Existing centres | Observation question |
|---|---:|---|
| C001 | 11 | Canopy identity, edge and alignment |
| C017 | 32 | Mixed/open-cover composition |
| C027 | 42 | Channel-margin cover and original quality gap |

RGB photography may suffice for the first two. Amphibious contact sensing is conditional on a separately justified current task. No hyperspectral payload, drone prototype, planting need, field access or comparative advantage is demonstrated.

The full source package includes `docs/DRONE_KHOR_KALBA_HANDOFF.md`, three inspection briefs and source attribution. All observation fields below are empty.

In [ ]:
import csv, hashlib, io, json, tempfile, zipfile
from pathlib import Path

# Embedded cached records; these are historical cell centres, NOT flight waypoints.
PAYLOADS = json.loads("{\"inspection_requests.json\":\"{\\n  \\\"schema_version\\\": \\\"1.0\\\",\\n  \\\"package_id\\\": \\\"MP-KHOR-KALBA-HANDOFF-01\\\",\\n  \\\"prepared_date\\\": \\\"2026-10-07\\\",\\n  \\\"evidence_status\\\": \\\"Cached historical evidence and proposed observation requirements; no inspection performed\\\",\\n  \\\"coordinate_semantics\\\": \\\"WGS84 original diagnostic cell centres, not route, waypoint, access, landing or contact sampling positions\\\",\\n  \\\"historical_satellite_dates\\\": [\\n    \\\"2018-11-20\\\",\\n    \\\"2018-11-30\\\"\\n  ],\\n  \\\"current_field_access\\\": \\\"unconfirmed\\\",\\n  \\\"sources\\\": {\\n    \\\"../audit/all_audit_cells.csv\\\": \\\"80a92cf778a92f8bdcc4a99e39dfe906f193a3eaad1dacbbd166f13481806e73\\\",\\n    \\\"../reference_context/case_comparison.json\\\": \\\"895116d01d0c79ce8191a63de503051304f58c32fe32238061ab962b92492f35\\\"\\n  },\\n  \\\"temporal_gate\\\": \\\"Current observations cannot directly validate 2018 cover; record actual dates and preserve unresolved historical truth\\\",\\n  \\\"cases\\\": [\\n    {\\n      \\\"case_id\\\": \\\"C001\\\",\\n      \\\"target_count\\\": 11,\\n      \\\"original_category\\\": \\\"Prediction positive / GMW not mapped\\\",\\n      \\\"cell_ids\\\": [\\n        \\\"C001_r88_c83\\\",\\n        \\\"C001_r89_c83\\\",\\n        \\\"C001_r90_c82\\\",\\n        \\\"C001_r91_c81\\\",\\n        \\\"C001_r92_c80\\\",\\n        \\\"C001_r92_c81\\\",\\n        \\\"C001_r93_c80\\\",\\n        \\\"C001_r93_c81\\\",\\n        \\\"C001_r94_c81\\\",\\n        \\\"C001_r95_c81\\\",\\n        \\\"C001_r96_c82\\\"\\n      ],\\n      \\\"reference_comparison_2018\\\": {\\n        \\\"both_not_mapped\\\": 4,\\n        \\\"v3_only_mapped\\\": 0,\\n        \\\"v4_only_mapped\\\": 7,\\n        \\\"both_mapped\\\": 0,\\n        \\\"centres\\\": 11,\\n        \\\"differently_mapped_centres\\\": 7,\\n        \\\"differently_mapped_fraction\\\": 0.6363636363636364,\\n        \\\"mapped_jaccard\\\": 0,\\n        \\\"interpretation\\\": \\\"Reference-map agreement, not field accuracy or ecological change\\\"\\n      },\\n      \\\"worldcover_2020_later_context\\\": [\\n        {\\n          \\\"code\\\": 95,\\n          \\\"class\\\": \\\"Mangroves\\\",\\n          \\\"centres\\\": 11\\n        }\\n      ],\\n      \\\"cell_truth\\\": \\\"unresolved\\\",\\n      \\\"question\\\": \\\"Which target cells and surroundings contain identifiable canopy at the observation date, and how certain is the edge alignment?\\\",\\n      \\\"request\\\": [\\n        \\\"Dated aligned cover evidence across cells and surrounding edge\\\",\\n        \\\"Qualified vegetation identification\\\",\\n        \\\"Visibility, occlusion and positioning uncertainty\\\"\\n      ],\\n      \\\"candidate_method\\\": \\\"Existing licensed imagery or safe shore/boat photography; ordinary RGB drone only if access and observation quality justify it\\\",\\n      \\\"amphibious_need\\\": \\\"Not established for photographic canopy-edge review\\\",\\n      \\\"deployment_status\\\": \\\"not_authorized_or_planned\\\",\\n      \\\"field_observations_collected\\\": 0,\\n      \\\"planting_recommendation\\\": \\\"none\\\"\\n    },\\n    {\\n      \\\"case_id\\\": \\\"C017\\\",\\n      \\\"target_count\\\": 32,\\n      \\\"original_category\\\": \\\"Prediction negative / GMW mapped\\\",\\n      \\\"cell_ids\\\": [\\n        \\\"C017_r64_c87\\\",\\n        \\\"C017_r64_c88\\\",\\n        \\\"C017_r64_c89\\\",\\n        \\\"C017_r65_c86\\\",\\n        \\\"C017_r65_c87\\\",\\n        \\\"C017_r65_c88\\\",\\n        \\\"C017_r65_c89\\\",\\n        \\\"C017_r65_c90\\\",\\n        \\\"C017_r66_c84\\\",\\n        \\\"C017_r66_c86\\\",\\n        \\\"C017_r66_c87\\\",\\n        \\\"C017_r66_c88\\\",\\n        \\\"C017_r67_c84\\\",\\n        \\\"C017_r67_c85\\\",\\n        \\\"C017_r67_c86\\\",\\n        \\\"C017_r67_c87\\\",\\n        \\\"C017_r68_c84\\\",\\n        \\\"C017_r68_c85\\\",\\n        \\\"C017_r68_c86\\\",\\n        \\\"C017_r68_c87\\\",\\n        \\\"C017_r69_c84\\\",\\n        \\\"C017_r69_c85\\\",\\n        \\\"C017_r69_c86\\\",\\n        \\\"C017_r69_c87\\\",\\n        \\\"C017_r70_c84\\\",\\n        \\\"C017_r70_c85\\\",\\n        \\\"C017_r71_c85\\\",\\n        \\\"C017_r72_c85\\\",\\n        \\\"C017_r73_c85\\\",\\n        \\\"C017_r74_c83\\\",\\n        \\\"C017_r74_c84\\\",\\n        \\\"C017_r75_c84\\\"\\n      ],\\n      \\\"reference_comparison_2018\\\": {\\n        \\\"both_not_mapped\\\": 0,\\n        \\\"v3_only_mapped\\\": 26,\\n        \\\"v4_only_mapped\\\": 0,\\n        \\\"both_mapped\\\": 6,\\n        \\\"centres\\\": 32,\\n        \\\"differently_mapped_centres\\\": 26,\\n        \\\"differently_mapped_fraction\\\": 0.8125,\\n        \\\"mapped_jaccard\\\": 0.1875,\\n        \\\"interpretation\\\": \\\"Reference-map agreement, not field accuracy or ecological change\\\"\\n      },\\n      \\\"worldcover_2020_later_context\\\": [\\n        {\\n          \\\"code\\\": 60,\\n          \\\"class\\\": \\\"Bare / sparse vegetation\\\",\\n          \\\"centres\\\": 7\\n        },\\n        {\\n          \\\"code\\\": 90,\\n          \\\"class\\\": \\\"Herbaceous wetland\\\",\\n          \\\"centres\\\": 4\\n        },\\n        {\\n          \\\"code\\\": 95,\\n          \\\"class\\\": \\\"Mangroves\\\",\\n          \\\"centres\\\": 21\\n        }\\n      ],\\n      \\\"cell_truth\\\": \\\"unresolved\\\",\\n      \\\"question\\\": \\\"What cover composition can be supported for each cell, including mixed vegetation, water and open ground?\\\",\\n      \\\"request\\\": [\\n        \\\"Cover categories with uncertainty\\\",\\n        \\\"Canopy/open-cover fractions only with adequate aligned coverage and documented labelling method\\\",\\n        \\\"Qualified vegetation identification\\\"\\n      ],\\n      \\\"candidate_method\\\": \\\"Detailed licensed imagery, shore/boat photography or ordinary RGB drone where justified\\\",\\n      \\\"amphibious_need\\\": \\\"Not established for photographic mixed-cover review\\\",\\n      \\\"deployment_status\\\": \\\"not_authorized_or_planned\\\",\\n      \\\"field_observations_collected\\\": 0,\\n      \\\"planting_recommendation\\\": \\\"none\\\"\\n    },\\n    {\\n      \\\"case_id\\\": \\\"C027\\\",\\n      \\\"target_count\\\": 42,\\n      \\\"original_category\\\": \\\"Quality-excluded / GMW mapped\\\",\\n      \\\"cell_ids\\\": [\\n        \\\"C027_r61_c96\\\",\\n        \\\"C027_r62_c96\\\",\\n        \\\"C027_r63_c96\\\",\\n        \\\"C027_r64_c96\\\",\\n        \\\"C027_r64_c97\\\",\\n        \\\"C027_r65_c95\\\",\\n        \\\"C027_r65_c97\\\",\\n        \\\"C027_r66_c96\\\",\\n        \\\"C027_r66_c97\\\",\\n        \\\"C027_r67_c96\\\",\\n        \\\"C027_r67_c97\\\",\\n        \\\"C027_r68_c96\\\",\\n        \\\"C027_r68_c97\\\",\\n        \\\"C027_r69_c96\\\",\\n        \\\"C027_r69_c97\\\",\\n        \\\"C027_r70_c96\\\",\\n        \\\"C027_r70_c97\\\",\\n        \\\"C027_r71_c96\\\",\\n        \\\"C027_r72_c95\\\",\\n        \\\"C027_r72_c96\\\",\\n        \\\"C027_r73_c95\\\",\\n        \\\"C027_r73_c96\\\",\\n        \\\"C027_r74_c95\\\",\\n        \\\"C027_r74_c96\\\",\\n        \\\"C027_r75_c96\\\",\\n        \\\"C027_r76_c96\\\",\\n        \\\"C027_r76_c97\\\",\\n        \\\"C027_r77_c96\\\",\\n        \\\"C027_r77_c97\\\",\\n        \\\"C027_r78_c97\\\",\\n        \\\"C027_r78_c98\\\",\\n        \\\"C027_r79_c97\\\",\\n        \\\"C027_r79_c98\\\",\\n        \\\"C027_r80_c97\\\",\\n        \\\"C027_r80_c98\\\",\\n        \\\"C027_r80_c99\\\",\\n        \\\"C027_r81_c98\\\",\\n        \\\"C027_r81_c99\\\",\\n        \\\"C027_r82_c99\\\",\\n        \\\"C027_r82_c100\\\",\\n        \\\"C027_r83_c100\\\",\\n        \\\"C027_r84_c100\\\"\\n      ],\\n      \\\"reference_comparison_2018\\\": {\\n        \\\"both_not_mapped\\\": 0,\\n        \\\"v3_only_mapped\\\": 6,\\n        \\\"v4_only_mapped\\\": 0,\\n        \\\"both_mapped\\\": 36,\\n        \\\"centres\\\": 42,\\n        \\\"differently_mapped_centres\\\": 6,\\n        \\\"differently_mapped_fraction\\\": 0.14285714285714285,\\n        \\\"mapped_jaccard\\\": 0.8571428571428571,\\n        \\\"interpretation\\\": \\\"Reference-map agreement, not field accuracy or ecological change\\\"\\n      },\\n      \\\"worldcover_2020_later_context\\\": [\\n        {\\n          \\\"code\\\": 10,\\n          \\\"class\\\": \\\"Tree cover\\\",\\n          \\\"centres\\\": 2\\n        },\\n        {\\n          \\\"code\\\": 20,\\n          \\\"class\\\": \\\"Shrubland\\\",\\n          \\\"centres\\\": 3\\n        },\\n        {\\n          \\\"code\\\": 80,\\n          \\\"class\\\": \\\"Permanent water\\\",\\n          \\\"centres\\\": 9\\n        },\\n        {\\n          \\\"code\\\": 95,\\n          \\\"class\\\": \\\"Mangroves\\\",\\n          \\\"centres\\\": 28\\n        }\\n      ],\\n      \\\"cell_truth\\\": \\\"unresolved\\\",\\n      \\\"question\\\": \\\"What dated cover evidence is available at the channel margin, and what limits the original usable satellite coverage?\\\",\\n      \\\"request\\\": [\\n        \\\"Historical per-cell quality/input review kept separate from physical causes\\\",\\n        \\\"Dated aligned vegetation/water/open-cover observations\\\",\\n        \\\"Visibility and safe-access assessment\\\"\\n      ],\\n      \\\"candidate_method\\\": \\\"Historical source review first; current channel-edge photography only as a separate workflow trial\\\",\\n      \\\"amphibious_need\\\": \\\"Conditional separate current contact-sensing hypothesis, not inferred from exclusions\\\",\\n      \\\"conditional_contact_task\\\": \\\"If a qualified practitioner establishes a water-contact question: conductivity/salinity, temperature, depth and calibration information at separately justified safe sampling sites; no measurements or instrument validation exist\\\",\\n      \\\"deployment_status\\\": \\\"not_authorized_or_planned\\\",\\n      \\\"field_observations_collected\\\": 0,\\n      \\\"planting_recommendation\\\": \\\"none\\\"\\n    }\\n  ]\\n}\\n\",\"inspection_targets.geojson\":\"{\\n  \\\"type\\\": \\\"FeatureCollection\\\",\\n  \\\"name\\\": \\\"Historical Khor Kalba inspection evidence targets - NOT flight waypoints\\\",\\n  \\\"features\\\": [\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r88_c83\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36652203304431,\\n          24.99018269207781\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"88\\\",\\n        \\\"column\\\": \\\"83\\\",\\n        \\\"longitude\\\": \\\"56.36652203304431\\\",\\n        \\\"latitude\\\": \\\"24.99018269207781\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r89_c83\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366522958699484,\\n          24.990002081080895\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"89\\\",\\n        \\\"column\\\": \\\"83\\\",\\n        \\\"longitude\\\": \\\"56.366522958699484\\\",\\n        \\\"latitude\\\": \\\"24.990002081080895\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r90_c82\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36632571585984,\\n          24.98982062631664\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"90\\\",\\n        \\\"column\\\": \\\"82\\\",\\n        \\\"longitude\\\": \\\"56.36632571585984\\\",\\n        \\\"latitude\\\": \\\"24.98982062631664\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r91_c81\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366128473599225,\\n          24.989639171297934\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"91\\\",\\n        \\\"column\\\": \\\"81\\\",\\n        \\\"longitude\\\": \\\"56.366128473599225\\\",\\n        \\\"latitude\\\": \\\"24.989639171297934\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r92_c80\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.365931231917656,\\n          24.989457716024788\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"92\\\",\\n        \\\"column\\\": \\\"80\\\",\\n        \\\"longitude\\\": \\\"56.365931231917656\\\",\\n        \\\"latitude\\\": \\\"24.989457716024788\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r92_c81\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36612939980653,\\n          24.989458560301614\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"92\\\",\\n        \\\"column\\\": \\\"81\\\",\\n        \\\"longitude\\\": \\\"56.36612939980653\\\",\\n        \\\"latitude\\\": \\\"24.989458560301614\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"4\\\",\\n        \\\"scl_nov30\\\": \\\"4\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r93_c80\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.3659321584055,\\n          24.989277105030975\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"93\\\",\\n        \\\"column\\\": \\\"80\\\",\\n        \\\"longitude\\\": \\\"56.3659321584055\\\",\\n        \\\"latitude\\\": \\\"24.989277105030975\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"agrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r93_c81\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366130326004864,\\n          24.98927794930088\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"93\\\",\\n        \\\"column\\\": \\\"81\\\",\\n        \\\"longitude\\\": \\\"56.366130326004864\\\",\\n        \\\"latitude\\\": \\\"24.98927794930088\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"4\\\",\\n        \\\"scl_nov30\\\": \\\"4\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r94_c81\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36613125219422,\\n          24.98909733829573\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"94\\\",\\n        \\\"column\\\": \\\"81\\\",\\n        \\\"longitude\\\": \\\"56.36613125219422\\\",\\n        \\\"latitude\\\": \\\"24.98909733829573\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"4\\\",\\n        \\\"scl_nov30\\\": \\\"4\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r95_c81\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36613217837461,\\n          24.988916727286167\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"95\\\",\\n        \\\"column\\\": \\\"81\\\",\\n        \\\"longitude\\\": \\\"56.36613217837461\\\",\\n        \\\"latitude\\\": \\\"24.988916727286167\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"4\\\",\\n        \\\"scl_nov30\\\": \\\"4\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C001_r96_c82\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366331271285844,\\n          24.98873696025746\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C001\\\",\\n        \\\"category_nov20\\\": \\\"predicted_positive_reference_negative\\\",\\n        \\\"row\\\": \\\"96\\\",\\n        \\\"column\\\": \\\"82\\\",\\n        \\\"longitude\\\": \\\"56.366331271285844\\\",\\n        \\\"latitude\\\": \\\"24.98873696025746\\\",\\n        \\\"gmw\\\": \\\"0\\\",\\n        \\\"prediction_nov20\\\": \\\"1\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"4\\\",\\n        \\\"scl_nov30\\\": \\\"4\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r64_c87\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.367292518777276,\\n          24.994520727810993\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"64\\\",\\n        \\\"column\\\": \\\"87\\\",\\n        \\\"longitude\\\": \\\"56.367292518777276\\\",\\n        \\\"latitude\\\": \\\"24.994520727810993\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 90,\\n        \\\"worldcover_class_2020\\\": \\\"Herbaceous wetland\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r64_c88\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.367490694836356,\\n          24.994521570434173\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"64\\\",\\n        \\\"column\\\": \\\"88\\\",\\n        \\\"longitude\\\": \\\"56.367490694836356\\\",\\n        \\\"latitude\\\": \\\"24.994521570434173\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"5\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 90,\\n        \\\"worldcover_class_2020\\\": \\\"Herbaceous wetland\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r64_c89\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.367688870904395,\\n          24.994522412793426\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"64\\\",\\n        \\\"column\\\": \\\"89\\\",\\n        \\\"longitude\\\": \\\"56.367688870904395\\\",\\n        \\\"latitude\\\": \\\"24.994522412793426\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r65_c86\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36709526772873,\\n          24.99433927401221\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"65\\\",\\n        \\\"column\\\": \\\"86\\\",\\n        \\\"longitude\\\": \\\"56.36709526772873\\\",\\n        \\\"latitude\\\": \\\"24.99433927401221\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r65_c87\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36729344348927,\\n          24.99434011689241\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"65\\\",\\n        \\\"column\\\": \\\"87\\\",\\n        \\\"longitude\\\": \\\"56.36729344348927\\\",\\n        \\\"latitude\\\": \\\"24.99434011689241\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 60,\\n        \\\"worldcover_class_2020\\\": \\\"Bare / sparse vegetation\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r65_c88\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.367491619258765,\\n          24.994340959508673\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"65\\\",\\n        \\\"column\\\": \\\"88\\\",\\n        \\\"longitude\\\": \\\"56.367491619258765\\\",\\n        \\\"latitude\\\": \\\"24.994340959508673\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"5\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 60,\\n        \\\"worldcover_class_2020\\\": \\\"Bare / sparse vegetation\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r65_c89\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36768979503721,\\n          24.99434180186102\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"65\\\",\\n        \\\"column\\\": \\\"89\\\",\\n        \\\"longitude\\\": \\\"56.36768979503721\\\",\\n        \\\"latitude\\\": \\\"24.99434180186102\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 90,\\n        \\\"worldcover_class_2020\\\": \\\"Herbaceous wetland\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r65_c90\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.3678879708246,\\n          24.994342643949434\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"65\\\",\\n        \\\"column\\\": \\\"90\\\",\\n        \\\"longitude\\\": \\\"56.3678879708246\\\",\\n        \\\"latitude\\\": \\\"24.994342643949434\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r66_c84\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36669984180633,\\n          24.99415697655777\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"66\\\",\\n        \\\"column\\\": \\\"84\\\",\\n        \\\"longitude\\\": \\\"56.36669984180633\\\",\\n        \\\"latitude\\\": \\\"24.99415697655777\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r66_c86\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.367096192721355,\\n          24.994158663096115\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"66\\\",\\n        \\\"column\\\": \\\"86\\\",\\n        \\\"longitude\\\": \\\"56.367096192721355\\\",\\n        \\\"latitude\\\": \\\"24.994158663096115\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 90,\\n        \\\"worldcover_class_2020\\\": \\\"Herbaceous wetland\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r66_c87\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.367294368192304,\\n          24.9941595059694\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"66\\\",\\n        \\\"column\\\": \\\"87\\\",\\n        \\\"longitude\\\": \\\"56.367294368192304\\\",\\n        \\\"latitude\\\": \\\"24.9941595059694\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"5\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 60,\\n        \\\"worldcover_class_2020\\\": \\\"Bare / sparse vegetation\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r66_c88\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36749254367221,\\n          24.99416034857876\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"66\\\",\\n        \\\"column\\\": \\\"88\\\",\\n        \\\"longitude\\\": \\\"56.36749254367221\\\",\\n        \\\"latitude\\\": \\\"24.99416034857876\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"5\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 60,\\n        \\\"worldcover_class_2020\\\": \\\"Bare / sparse vegetation\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r67_c84\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36670076736917,\\n          24.993976365651086\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"67\\\",\\n        \\\"column\\\": \\\"84\\\",\\n        \\\"longitude\\\": \\\"56.36670076736917\\\",\\n        \\\"latitude\\\": \\\"24.993976365651086\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r67_c85\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366898942532615,\\n          24.99397720904531\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"67\\\",\\n        \\\"column\\\": \\\"85\\\",\\n        \\\"longitude\\\": \\\"56.366898942532615\\\",\\n        \\\"latitude\\\": \\\"24.99397720904531\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r67_c86\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36709711770502,\\n          24.99397805217561\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"67\\\",\\n        \\\"column\\\": \\\"86\\\",\\n        \\\"longitude\\\": \\\"56.36709711770502\\\",\\n        \\\"latitude\\\": \\\"24.99397805217561\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 60,\\n        \\\"worldcover_class_2020\\\": \\\"Bare / sparse vegetation\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r67_c87\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36729529288639,\\n          24.993978895041977\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"67\\\",\\n        \\\"column\\\": \\\"87\\\",\\n        \\\"longitude\\\": \\\"56.36729529288639\\\",\\n        \\\"latitude\\\": \\\"24.993978895041977\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"5\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 60,\\n        \\\"worldcover_class_2020\\\": \\\"Bare / sparse vegetation\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r68_c84\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366701692923044,\\n          24.993795754739992\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"68\\\",\\n        \\\"column\\\": \\\"84\\\",\\n        \\\"longitude\\\": \\\"56.366701692923044\\\",\\n        \\\"latitude\\\": \\\"24.993795754739992\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r68_c85\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.3668998677969,\\n          24.9937965981273\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"68\\\",\\n        \\\"column\\\": \\\"85\\\",\\n        \\\"longitude\\\": \\\"56.3668998677969\\\",\\n        \\\"latitude\\\": \\\"24.9937965981273\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r68_c86\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36709804267973,\\n          24.993797441250685\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"68\\\",\\n        \\\"column\\\": \\\"86\\\",\\n        \\\"longitude\\\": \\\"56.36709804267973\\\",\\n        \\\"latitude\\\": \\\"24.993797441250685\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r68_c87\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.367296217571514,\\n          24.993798284110152\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"68\\\",\\n        \\\"column\\\": \\\"87\\\",\\n        \\\"longitude\\\": \\\"56.367296217571514\\\",\\n        \\\"latitude\\\": \\\"24.993798284110152\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"5\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 60,\\n        \\\"worldcover_class_2020\\\": \\\"Bare / sparse vegetation\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r69_c84\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366702618467954,\\n          24.993615143824478\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"69\\\",\\n        \\\"column\\\": \\\"84\\\",\\n        \\\"longitude\\\": \\\"56.366702618467954\\\",\\n        \\\"latitude\\\": \\\"24.993615143824478\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r69_c85\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36690079305223,\\n          24.993615987204866\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"69\\\",\\n        \\\"column\\\": \\\"85\\\",\\n        \\\"longitude\\\": \\\"56.36690079305223\\\",\\n        \\\"latitude\\\": \\\"24.993615987204866\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r69_c86\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36709896764548,\\n          24.99361683032134\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"69\\\",\\n        \\\"column\\\": \\\"86\\\",\\n        \\\"longitude\\\": \\\"56.36709896764548\\\",\\n        \\\"latitude\\\": \\\"24.99361683032134\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r69_c87\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.367297142247686,\\n          24.99361767317389\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"69\\\",\\n        \\\"column\\\": \\\"87\\\",\\n        \\\"longitude\\\": \\\"56.367297142247686\\\",\\n        \\\"latitude\\\": \\\"24.99361767317389\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r70_c84\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.3667035440039,\\n          24.993434532904548\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"70\\\",\\n        \\\"column\\\": \\\"84\\\",\\n        \\\"longitude\\\": \\\"56.3667035440039\\\",\\n        \\\"latitude\\\": \\\"24.993434532904548\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r70_c85\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36690171829861,\\n          24.993435376278022\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"70\\\",\\n        \\\"column\\\": \\\"85\\\",\\n        \\\"longitude\\\": \\\"56.36690171829861\\\",\\n        \\\"latitude\\\": \\\"24.993435376278022\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r71_c85\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366902643536015,\\n          24.99325476534676\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"71\\\",\\n        \\\"column\\\": \\\"85\\\",\\n        \\\"longitude\\\": \\\"56.366902643536015\\\",\\n        \\\"latitude\\\": \\\"24.99325476534676\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r72_c85\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36690356876446,\\n          24.99307415441108\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"72\\\",\\n        \\\"column\\\": \\\"85\\\",\\n        \\\"longitude\\\": \\\"56.36690356876446\\\",\\n        \\\"latitude\\\": \\\"24.99307415441108\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r73_c85\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36690449398394,\\n          24.99289354347099\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"73\\\",\\n        \\\"column\\\": \\\"85\\\",\\n        \\\"longitude\\\": \\\"56.36690449398394\\\",\\n        \\\"latitude\\\": \\\"24.99289354347099\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r74_c83\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36650907293057,\\n          24.992711245570945\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"74\\\",\\n        \\\"column\\\": \\\"83\\\",\\n        \\\"longitude\\\": \\\"56.36650907293057\\\",\\n        \\\"latitude\\\": \\\"24.992711245570945\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r74_c84\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.366707246058034,\\n          24.992712089180667\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"74\\\",\\n        \\\"column\\\": \\\"84\\\",\\n        \\\"longitude\\\": \\\"56.366707246058034\\\",\\n        \\\"latitude\\\": \\\"24.992712089180667\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C017_r75_c84\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36670817154916,\\n          24.992531478238657\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C017\\\",\\n        \\\"category_nov20\\\": \\\"predicted_negative_reference_positive\\\",\\n        \\\"row\\\": \\\"75\\\",\\n        \\\"column\\\": \\\"84\\\",\\n        \\\"longitude\\\": \\\"56.36670817154916\\\",\\n        \\\"latitude\\\": \\\"24.992531478238657\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"0\\\",\\n        \\\"prediction_nov30\\\": \\\"0\\\",\\n        \\\"status_nov30\\\": \\\"disagrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"6\\\",\\n        \\\"scl_nov30\\\": \\\"6\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r61_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369073337260474,\\n          24.99507013483367\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"61\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369073337260474\\\",\\n        \\\"latitude\\\": \\\"24.99507013483367\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 20,\\n        \\\"worldcover_class_2020\\\": \\\"Shrubland\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r62_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369074259392946,\\n          24.994889523866235\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"62\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369074259392946\\\",\\n        \\\"latitude\\\": \\\"24.994889523866235\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r63_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36907518151648,\\n          24.994708912894378\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"63\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.36907518151648\\\",\\n        \\\"latitude\\\": \\\"24.994708912894378\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"7\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r64_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36907610363109,\\n          24.994528301918113\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"64\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.36907610363109\\\",\\n        \\\"latitude\\\": \\\"24.994528301918113\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"7\\\",\\n        \\\"scl_nov30\\\": \\\"7\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r64_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36927427977064,\\n          24.994529142165916\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"64\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.36927427977064\\\",\\n        \\\"latitude\\\": \\\"24.994529142165916\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r65_c95\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36887884989573,\\n          24.994346850432574\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"65\\\",\\n        \\\"column\\\": \\\"95\\\",\\n        \\\"longitude\\\": \\\"56.36887884989573\\\",\\n        \\\"latitude\\\": \\\"24.994346850432574\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"agrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"7\\\",\\n        \\\"scl_nov30\\\": \\\"4\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r65_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36927520158673,\\n          24.994348531178336\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"65\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.36927520158673\\\",\\n        \\\"latitude\\\": \\\"24.994348531178336\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r66_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36907794783351,\\n          24.994167079952316\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"66\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.36907794783351\\\",\\n        \\\"latitude\\\": \\\"24.994167079952316\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"2\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r66_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369276123393895,\\n          24.994167920186342\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"66\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.369276123393895\\\",\\n        \\\"latitude\\\": \\\"24.994167920186342\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r67_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369078869921324,\\n          24.99398646896279\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"67\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369078869921324\\\",\\n        \\\"latitude\\\": \\\"24.99398646896279\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r67_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36927704519212,\\n          24.99398730918994\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"67\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.36927704519212\\\",\\n        \\\"latitude\\\": \\\"24.99398730918994\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r68_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369079792000214,\\n          24.993805857968866\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"68\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369079792000214\\\",\\n        \\\"latitude\\\": \\\"24.993805857968866\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"7\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r68_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36927796698143,\\n          24.993806698189115\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"68\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.36927796698143\\\",\\n        \\\"latitude\\\": \\\"24.993806698189115\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r69_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36908071407017,\\n          24.993625246970506\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"69\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.36908071407017\\\",\\n        \\\"latitude\\\": \\\"24.993625246970506\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r69_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369278888761805,\\n          24.993626087183873\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"69\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.369278888761805\\\",\\n        \\\"latitude\\\": \\\"24.993626087183873\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r70_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.3690816361312,\\n          24.99344463596774\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"70\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.3690816361312\\\",\\n        \\\"latitude\\\": \\\"24.99344463596774\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r70_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36927981053325,\\n          24.993445476174216\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"70\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.36927981053325\\\",\\n        \\\"latitude\\\": \\\"24.993445476174216\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r71_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369082558183294,\\n          24.993264024960563\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"71\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369082558183294\\\",\\n        \\\"latitude\\\": \\\"24.993264024960563\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 10,\\n        \\\"worldcover_class_2020\\\": \\\"Tree cover\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r72_c95\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36888530641248,\\n          24.99308257349235\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"72\\\",\\n        \\\"column\\\": \\\"95\\\",\\n        \\\"longitude\\\": \\\"56.36888530641248\\\",\\n        \\\"latitude\\\": \\\"24.99308257349235\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"7\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r72_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36908348022646,\\n          24.99308341394896\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"72\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.36908348022646\\\",\\n        \\\"latitude\\\": \\\"24.99308341394896\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r73_c95\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.368886228736294,\\n          24.992901962483234\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"73\\\",\\n        \\\"column\\\": \\\"95\\\",\\n        \\\"longitude\\\": \\\"56.368886228736294\\\",\\n        \\\"latitude\\\": \\\"24.992901962483234\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r73_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369084402260704,\\n          24.992902802932946\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"73\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369084402260704\\\",\\n        \\\"latitude\\\": \\\"24.992902802932946\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 10,\\n        \\\"worldcover_class_2020\\\": \\\"Tree cover\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r74_c95\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.368887151051155,\\n          24.992721351469697\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"74\\\",\\n        \\\"column\\\": \\\"95\\\",\\n        \\\"longitude\\\": \\\"56.368887151051155\\\",\\n        \\\"latitude\\\": \\\"24.992721351469697\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r74_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369085324286004,\\n          24.99272219191252\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"74\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369085324286004\\\",\\n        \\\"latitude\\\": \\\"24.99272219191252\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r75_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36908624630238,\\n          24.99254158088767\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"75\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.36908624630238\\\",\\n        \\\"latitude\\\": \\\"24.99254158088767\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r76_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369087168309825,\\n          24.99236096985841\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"76\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369087168309825\\\",\\n        \\\"latitude\\\": \\\"24.99236096985841\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r76_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36928534097448,\\n          24.99236181002355\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"76\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.36928534097448\\\",\\n        \\\"latitude\\\": \\\"24.99236181002355\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r77_c96\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369088090308345,\\n          24.992180358824736\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"77\\\",\\n        \\\"column\\\": \\\"96\\\",\\n        \\\"longitude\\\": \\\"56.369088090308345\\\",\\n        \\\"latitude\\\": \\\"24.992180358824736\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r77_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369286262683445,\\n          24.99218119898299\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"77\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.369286262683445\\\",\\n        \\\"latitude\\\": \\\"24.99218119898299\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r78_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369287184383474,\\n          24.99200058793801\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"78\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.369287184383474\\\",\\n        \\\"latitude\\\": \\\"24.99200058793801\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"3\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r78_c98\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36948535647794,\\n          24.992001427825464\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"78\\\",\\n        \\\"column\\\": \\\"98\\\",\\n        \\\"longitude\\\": \\\"56.36948535647794\\\",\\n        \\\"latitude\\\": \\\"24.992001427825464\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r79_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36928810607458,\\n          24.991819976888614\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"79\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.36928810607458\\\",\\n        \\\"latitude\\\": \\\"24.991819976888614\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"7\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r79_c98\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369486277879496,\\n          24.991820816769184\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"79\\\",\\n        \\\"column\\\": \\\"98\\\",\\n        \\\"longitude\\\": \\\"56.369486277879496\\\",\\n        \\\"latitude\\\": \\\"24.991820816769184\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 0,\\n        \\\"worldcover_v100_2020\\\": 20,\\n        \\\"worldcover_class_2020\\\": \\\"Shrubland\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r80_c97\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369289027756764,\\n          24.99163936583479\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"80\\\",\\n        \\\"column\\\": \\\"97\\\",\\n        \\\"longitude\\\": \\\"56.369289027756764\\\",\\n        \\\"latitude\\\": \\\"24.99163936583479\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"1\\\",\\n        \\\"status_nov30\\\": \\\"agrees_with_reference\\\",\\n        \\\"scl_nov20\\\": \\\"7\\\",\\n        \\\"scl_nov30\\\": \\\"4\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r80_c98\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36948719927212,\\n          24.991640205708485\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"80\\\",\\n        \\\"column\\\": \\\"98\\\",\\n        \\\"longitude\\\": \\\"56.36948719927212\\\",\\n        \\\"latitude\\\": \\\"24.991640205708485\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 20,\\n        \\\"worldcover_class_2020\\\": \\\"Shrubland\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r80_c99\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36968537079642,\\n          24.99164104531827\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"80\\\",\\n        \\\"column\\\": \\\"99\\\",\\n        \\\"longitude\\\": \\\"56.36968537079642\\\",\\n        \\\"latitude\\\": \\\"24.99164104531827\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 80,\\n        \\\"worldcover_class_2020\\\": \\\"Permanent water\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r81_c98\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36948812065583,\\n          24.991459594643363\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"81\\\",\\n        \\\"column\\\": \\\"98\\\",\\n        \\\"longitude\\\": \\\"56.36948812065583\\\",\\n        \\\"latitude\\\": \\\"24.991459594643363\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"7\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"False\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r81_c99\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36968629189058,\\n          24.991460434246275\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"81\\\",\\n        \\\"column\\\": \\\"99\\\",\\n        \\\"longitude\\\": \\\"56.36968629189058\\\",\\n        \\\"latitude\\\": \\\"24.991460434246275\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"3\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r82_c99\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36968721297582,\\n          24.991279823169865\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"82\\\",\\n        \\\"column\\\": \\\"99\\\",\\n        \\\"longitude\\\": \\\"56.36968721297582\\\",\\n        \\\"latitude\\\": \\\"24.991279823169865\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"2\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r82_c100\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36988538392994,\\n          24.991280662501993\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"82\\\",\\n        \\\"column\\\": \\\"100\\\",\\n        \\\"longitude\\\": \\\"56.36988538392994\\\",\\n        \\\"latitude\\\": \\\"24.991280662501993\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"2\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r83_c100\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.369886304716715,\\n          24.99110005141428\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"83\\\",\\n        \\\"column\\\": \\\"100\\\",\\n        \\\"longitude\\\": \\\"56.369886304716715\\\",\\n        \\\"latitude\\\": \\\"24.99110005141428\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"2\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    },\\n    {\\n      \\\"type\\\": \\\"Feature\\\",\\n      \\\"id\\\": \\\"C027_r84_c100\\\",\\n      \\\"geometry\\\": {\\n        \\\"type\\\": \\\"Point\\\",\\n        \\\"coordinates\\\": [\\n          56.36988722549458,\\n          24.99091944032215\\n        ]\\n      },\\n      \\\"properties\\\": {\\n        \\\"cluster_id\\\": \\\"C027\\\",\\n        \\\"category_nov20\\\": \\\"excluded_reference_positive\\\",\\n        \\\"row\\\": \\\"84\\\",\\n        \\\"column\\\": \\\"100\\\",\\n        \\\"longitude\\\": \\\"56.36988722549458\\\",\\n        \\\"latitude\\\": \\\"24.99091944032215\\\",\\n        \\\"gmw\\\": \\\"1\\\",\\n        \\\"prediction_nov20\\\": \\\"255\\\",\\n        \\\"prediction_nov30\\\": \\\"255\\\",\\n        \\\"status_nov30\\\": \\\"excluded\\\",\\n        \\\"scl_nov20\\\": \\\"2\\\",\\n        \\\"scl_nov30\\\": \\\"2\\\",\\n        \\\"adjacent_to_reference_transition\\\": \\\"True\\\",\\n        \\\"review_note\\\": \\\"\\\",\\n        \\\"cause\\\": \\\"unverified\\\",\\n        \\\"gmw_v4112_2018\\\": 1,\\n        \\\"worldcover_v100_2020\\\": 95,\\n        \\\"worldcover_class_2020\\\": \\\"Mangroves\\\",\\n        \\\"target_semantics\\\": \\\"original_historical_cell_centre_not_flight_waypoint\\\",\\n        \\\"field_observation_status\\\": \\\"not_collected\\\"\\n      }\\n    }\\n  ]\\n}\\n\",\"observation_return_template.json\":\"{\\n  \\\"schema_version\\\": \\\"1.0\\\",\\n  \\\"package_id\\\": \\\"MP-KHOR-KALBA-HANDOFF-01\\\",\\n  \\\"status\\\": \\\"unfilled\\\",\\n  \\\"observation_record_template\\\": {\\n    \\\"case_id\\\": null,\\n    \\\"cell_id\\\": null,\\n    \\\"acquisition_datetime_with_timezone\\\": null,\\n    \\\"actual_observer_longitude\\\": null,\\n    \\\"actual_observer_latitude\\\": null,\\n    \\\"position_uncertainty_m\\\": null,\\n    \\\"method\\\": null,\\n    \\\"sensor_and_calibration\\\": null,\\n    \\\"visibility_and_occlusion\\\": null,\\n    \\\"evidence_file\\\": null,\\n    \\\"evidence_redistribution_permission\\\": null,\\n    \\\"qualified_cover_identification\\\": null,\\n    \\\"cover_fraction_if_supported\\\": null,\\n    \\\"alignment_evidence\\\": null,\\n    \\\"temporal_relevance\\\": null,\\n    \\\"quality_decision\\\": null,\\n    \\\"quality_reason\\\": null,\\n    \\\"reviewer\\\": null,\\n    \\\"interpretation\\\": null,\\n    \\\"remaining_uncertainty\\\": null\\n  },\\n  \\\"cases\\\": [\\n    {\\n      \\\"case_id\\\": \\\"C001\\\",\\n      \\\"status\\\": \\\"not_collected\\\",\\n      \\\"observations\\\": [],\\n      \\\"conclusion\\\": null,\\n      \\\"independent_2018_truth\\\": \\\"unresolved\\\"\\n    },\\n    {\\n      \\\"case_id\\\": \\\"C017\\\",\\n      \\\"status\\\": \\\"not_collected\\\",\\n      \\\"observations\\\": [],\\n      \\\"conclusion\\\": null,\\n      \\\"independent_2018_truth\\\": \\\"unresolved\\\"\\n    },\\n    {\\n      \\\"case_id\\\": \\\"C027\\\",\\n      \\\"status\\\": \\\"not_collected\\\",\\n      \\\"observations\\\": [],\\n      \\\"conclusion\\\": null,\\n      \\\"independent_2018_truth\\\": \\\"unresolved\\\"\\n    }\\n  ]\\n}\\n\"}")
requests = json.loads(PAYLOADS['inspection_requests.json'])
targets = json.loads(PAYLOADS['inspection_targets.geojson'])
returns = json.loads(PAYLOADS['observation_return_template.json'])
print(requests['evidence_status'])
for case in requests['cases']:
    print(case['case_id'], case['target_count'], 'centres:', case['question'])


## Validate the handoff format

Counts, identifiers, copied coordinates and blank returns are checked before export. These checks are not new model evaluation or independent verification of the source maps.

In [ ]:
# Pure format and internal-consistency checks; no scientific processing.
expected_counts = {'C001': 11, 'C017': 32, 'C027': 42}
features = targets['features']
ids = [f['id'] for f in features]
assert targets['type'] == 'FeatureCollection'
assert len(features) == len(set(ids)) == 85
assert {c['case_id']: c['target_count'] for c in requests['cases']} == expected_counts
assert requests['historical_satellite_dates'] == ['2018-11-20', '2018-11-30']
for c in requests['cases']:
    selected = [f for f in features if f['properties']['cluster_id'] == c['case_id']]
    assert len(selected) == expected_counts[c['case_id']]
    assert c['cell_ids'] == [f['id'] for f in selected]
    assert c['field_observations_collected'] == 0 and c['cell_truth'] == 'unresolved'
    assert c['planting_recommendation'] == 'none'
    ref = c['reference_comparison_2018']
    assert sum(ref[k] for k in ['both_not_mapped', 'v3_only_mapped', 'v4_only_mapped', 'both_mapped']) == len(selected)
    assert sum(v['centres'] for v in c['worldcover_2020_later_context']) == len(selected)
for f in features:
    p = f['properties']
    assert f['geometry']['type'] == 'Point'
    assert f['geometry']['coordinates'] == [float(p['longitude']), float(p['latitude'])]
    assert f['id'] == f"{p['cluster_id']}_r{p['row']}_c{p['column']}"
    assert p['target_semantics'] == 'original_historical_cell_centre_not_flight_waypoint'
    assert p['field_observation_status'] == 'not_collected'
assert returns['status'] == 'unfilled'
assert all(v is None for v in returns['observation_record_template'].values())
assert all(not c['observations'] and c['conclusion'] is None for c in returns['cases'])
print('PASS: 3 cases / 85 unique cached centres / blank observation returns.')
print('This is internal package validation, not field validation or an approved mission.')


## Download the handoff ZIP

Includes observation requests, GeoJSON, an empty JSON return template, an 85-row blank CSV, attribution and a payload hash manifest. Do not upload the GeoJSON to an autopilot. Optional visual review in geojson.io does not validate cover, permission or safe access.

In [ ]:
# Export into a new Colab temporary folder; original uploads/files are never touched.
out = Path(tempfile.mkdtemp(prefix='mangrove_pulse_handoff_'))
for name, content in PAYLOADS.items():
    (out / name).write_text(content, encoding='utf-8')
columns = ['case_id', 'cell_id', 'target_longitude', 'target_latitude',
           'observation_status'] + list(returns['observation_record_template'])
columns = list(dict.fromkeys(columns))
stream = io.StringIO(newline='')
writer = csv.DictWriter(stream, fieldnames=columns)
writer.writeheader()
for f in features:
    writer.writerow({'case_id': f['properties']['cluster_id'], 'cell_id': f['id'],
                     'target_longitude': f['geometry']['coordinates'][0],
                     'target_latitude': f['geometry']['coordinates'][1],
                     'observation_status': 'not_collected'})
(out / 'observations_UNFILLED.csv').write_text(stream.getvalue(), encoding='utf-8')
(out / 'README.md').write_text(
    '# Historical Khor Kalba observation requests\n\n'
    'Three cases and 85 original satellite diagnostic cell centres.\n'
    'NOT a flight plan, access permission or ecological intervention recommendation.\n'
    'No field observations collected. Actual sensor position and time must be recorded separately.\n'
    'Current observations cannot directly validate November 2018 cover.\n'
    'Contains modified Copernicus Sentinel data (2018).\n'
    'GMW v3/v4.1.12 and WorldCover 2020 derived context: CC BY 4.0; see ATTRIBUTION.md.\n',
    encoding='utf-8')
(out / 'ATTRIBUTION.md').write_text(
    'GMW v3: Bunting et al. (2022), https://doi.org/10.5281/zenodo.6894273\n'
    'GMW v4.1.12: Bunting et al. (2026), https://doi.org/10.5281/zenodo.21346457\n'
    'ESA WorldCover 2020 v100: Zanaga et al. (2021), https://doi.org/10.5281/zenodo.5571936\n'
    'Data attribution: CC BY 4.0, https://creativecommons.org/licenses/by/4.0/\n'
    '© ESA WorldCover project 2020 / Contains modified Copernicus Sentinel data (2020) '
    'processed by ESA WorldCover consortium.\n'
    'Modifications: previously sampled reference labels and cached audit records selected '
    'and reformatted as observation requests and original-centre GeoJSON points. '
    'No new sampling, restricted imagery or provider endorsement.\n'
    'Original team code has no selected blanket open-source licence.\n',
    encoding='utf-8')
manifest = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(out.iterdir())}
(out / 'SHA256SUMS.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
archive = out.parent / (out.name + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out.iterdir()):
        z.write(p, p.name)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    saved = json.loads(z.read('SHA256SUMS.json'))
    assert set(z.namelist()) == set(saved) | {'SHA256SUMS.json'}
    assert all(hashlib.sha256(z.read(n)).hexdigest() == h for n, h in saved.items())
print('Validated export:', archive.name)
print('ZIP SHA-256:', hashlib.sha256(archive.read_bytes()).hexdigest())
try:
    from google.colab import files
except ImportError:
    print('Not in Colab; archive is available at', archive)
else:
    files.download(str(archive))
